# Markov Model testing

import and brief inspection of the data

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('data/2024_Wimbledon_featured_matches.csv')
df.head()

In [ ]:
pts_by_game = df.groupby('match_id').point_victor.value_counts().unstack()
pts_by_game['p1_win_rate'] = pts_by_game[1] / pts_by_game.sum(axis=1)
print(pts_by_game)

In [ ]:
pts_by_game['p1_win_rate'].describe()

Test the model:
P(player X will win a randomly sampled point in Y game | player X won a randomly sampled point in Y) = 
P(winner of the last point is player 1) * P(Player 1 wins a point) + P(winner of last point is player 2)*P(player 2 wins a point) = 
p1_win_rate * p1_win_rate + (1-p1_win_rate)*(1-p1_win_rate)

Thus any markov model of the form: Pn = f(P_(n-1)) must beat that eqn

In [ ]:
baselineFirstOrder = pts_by_game['p1_win_rate'] ** 2 + (1-pts_by_game['p1_win_rate'])**2
baselineFirstOrder.head()

In [ ]:
baselineFirstOrder.describe()

That's a pretty low bar!
51%

In [ ]:
df['prev_point_victor'] = df.groupby('match_id')['point_victor'].shift(1)
df[['point_victor', 'prev_point_victor']].head()

interior_pts = df.dropna(subset='prev_point_victor')
same_player_wins = (interior_pts['point_victor'] == interior_pts['prev_point_victor']).mean()
same_player_wins

The probability that a player X wins the next point given that they won the previous point is 53%, so does that mean that momentum has an effect?
Well maybe not, it's also possible that who is serving could make a large difference here.

In [ ]:
from sklearn.linear_model import LogisticRegression

interior_pts['momentum'] = (interior_pts['point_victor'] == interior_pts['prev_point_victor']).astype(int)

X = interior_pts[['momentum', 'server']]
Y = interior_pts[['point_victor']]

# No vectors contain any NAs
#print(X.isna().any())
#print(Y.isna().any())

model = LogisticRegression()
model.fit(X, Y)

Okay, so it seems that that the winner of the last point does not predict the winner of the next point!
In fact, it seems as though any "momentum" observed in the previous test is actually just a result of the previous
point winner encoding who was the more likely the previous server. 

# A2 Higher order Reccurence relation

Hypothesis: The winner of the next point can be predicted better with more data

Lets try predicting the next point with data from the last n=3 points

In [ ]:
from itertools import product

permute = True
lag = 3

def gen_k_lag(k:int):
    for i in range(1, k+1):
        # do not center the lag columns because that breaks how the cartesian product works
        df[f'prev_victor_{i}'] = (df.groupby('match_id')['point_victor'].shift(i)-1)
        if not permute:
            # center the lag columns because they will go directly into the model
            df[f'prev_victor_{i}'] -= 0.5

gen_k_lag(3)

df['server_centered'] = (df['server'] - 1.5) * 2

lag_cols = [f'prev_victor_{i}' for i in range(1, lag+1)]

if permute:
    for i, combo in enumerate(product([0, 1], repeat=len(lag_cols))):
        df[f'perm_{i}'] = (df[lag_cols] == list(combo)).all(axis=1).astype(int)

    features = [f'perm_{i}' for i in range(2 ** len(lag_cols))]
else:
    features = lag_cols
features.append('server_centered')

df_clean = df.dropna(subset=lag_cols)

X = df_clean[features]
y = df_clean['point_victor']

model = LogisticRegression()
model.fit(X, y)

See the distribution of each permutation


In [ ]:
df_clean[features].mean()

In [ ]:
df_clean[features]

In [ ]:
from sklearn.model_selection import cross_val_score
rec_scores = cross_val_score(model, X, y, cv=5, scoring='neg_log_loss')
print(rec_scores.mean())

In [ ]:
X = df_clean[['server_centered']]

baseline = LogisticRegression()
baseline.fit(X, y)

In [ ]:
baseline_scores = cross_val_score(baseline, X, y, cv=5, scoring='neg_log_loss')
print(baseline_scores.mean())
print(rec_scores.mean() - baseline_scores.mean())

In [ ]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(rec_scores, baseline_scores)
print(f"t-stat: {t_stat:.4f}")
print(f"p-value: {p_value:.8f}")

# Reccurence relation for service points

P(player X will will his next service point | player X won the last service point)


In [ ]:
df_service = df.copy()[['match_id', 'server', 'point_victor', 'game_no']]
df_service['server_won'] = df_service['server'] == df_service['point_victor']
df_service = df_service.sort_values(by=['match_id', 'server'])

df_service['prev_1_server_won'] = df_service.groupby(['match_id', 'server'])['server_won'].shift(1)
df_service['prev_2_server_won'] = df_service.groupby(['match_id', 'server'])['server_won'].shift(2)
df_service['prev_3_server_won'] = df_service.groupby(['match_id', 'server'])['server_won'].shift(3)
df_service = df_service.dropna()
df_service


In [ ]:
features = ['prev_1_server_won', 'prev_2_server_won', 'prev_3_server_won']
X = df_service[features]
y = df_service['server_won']

model = LogisticRegression(fit_intercept=False)
model.fit(X, y)

In [ ]:
rec_scores = cross_val_score(model, X, y, cv=10, scoring='neg_log_loss')
print(rec_scores.mean())

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='prior')
dummy.fit(X, y)

In [ ]:
dummy_scores = cross_val_score(dummy, X, y, cv=10, scoring='neg_log_loss')
print(dummy_scores.mean())

print(rec_scores.mean() - dummy_scores.mean())

In [ ]:
from sklearn.model_selection import permutation_test_score
score, perm_scores, p_value = permutation_test_score(
    model, X, y,
    scoring="neg_log_loss",   # log loss for your use case
    n_permutations=1000,
    cv=5,                     # cross-validate for robustness
    random_state=42
)

this test implies that the model is accurate

In [ ]:
print(p_value)

this test implies that although the model is accurate, that's only because it learned the distribution of points won by the server

In [ ]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(rec_scores, dummy_scores)
print(f"P-value: {p_value:.4f}")

TODO

Graphing\
Shuffling\
markov\
intercept\
Game momentum